# カフェBGM生成 (MusicGen)

週1回、このノートブックを上から順に実行するだけで、短いBGMクリップを複数生成し、
Google Driveの `bgm_clips` フォルダに保存します。

**使い方**
1. 上部メニュー「ランタイム」→「ランタイムのタイプを変更」→ GPU (T4) を選択
2. セルを上から順に実行 (Shift+Enter)
3. 生成が終わったら閉じてOK。あとはGitHub Actionsが自動で拾いに行きます

## 1. セットアップ

In [ ]:
!pip install -q audiocraft

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/bgm_clips'
os.makedirs(DRIVE_DIR, exist_ok=True)

## 2. モデル読み込み

In [ ]:
from audiocraft.models import MusicGen
from audiocraft.data.audio import audio_write

# 'small' は速いが品質はやや控えめ。品質重視なら 'medium' に変更可(生成が遅くなる)
model = MusicGen.get_pretrained('facebook/musicgen-small')

# 1回の生成の長さ(秒)。MusicGenの実用上の上限はおよそ30秒前後
model.set_generation_params(duration=30)

## 3. プロンプト設定

毎週このセルのプロンプトを少し変えると、チャンネル全体がマンネリ化しません。
同じ雰囲気で数パターン作るのがコツです。

In [ ]:
prompts = [
    "lofi cafe jazz, relaxing piano, soft brushed drums, warm vinyl texture",
    "lofi cafe jazz, mellow saxophone, gentle bassline, rainy afternoon mood",
    "lofi cafe jazz, soft electric piano, slow tempo, cozy morning atmosphere",
    "lofi cafe jazz, acoustic guitar, light percussion, calm and dreamy",
    "lofi cafe jazz, warm rhodes chords, subtle vinyl crackle, late night cafe",
]

## 4. 生成 & Driveに保存

In [ ]:
import datetime

today = datetime.date.today().isoformat()
batch_dir = os.path.join(DRIVE_DIR, today)
os.makedirs(batch_dir, exist_ok=True)

wav_outputs = model.generate(prompts, progress=True)

for i, audio in enumerate(wav_outputs):
    path = os.path.join(batch_dir, f'clip_{i}')
    audio_write(path, audio.cpu(), model.sample_rate, strategy='loudness')
    print(f'保存しました: {path}.wav')

print(f'\n完了。Drive上のフォルダ: bgm_clips/{today}')

## 5. 最新フォルダを 'latest' としてマーク

GitHub Actions側は常に `bgm_clips/latest` を参照しに行くだけにするため、
ここで今回作ったフォルダをコピーしておきます。

In [ ]:
import shutil

latest_dir = os.path.join(DRIVE_DIR, 'latest')
if os.path.exists(latest_dir):
    shutil.rmtree(latest_dir)
shutil.copytree(batch_dir, latest_dir)

print('latest フォルダを更新しました。')